# ComfyUI Fixed Colab

Run cells from top to bottom. This fixed notebook installs or updates ComfyUI on Google Drive, installs ComfyUI-Manager, fixes dependency issues like `alembic`, removes stale files that can cause `comfy_aimdo` import errors, downloads SD1.5 + VAE, and launches ComfyUI through Cloudflare Tunnel.

## 1. Check GPU

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=False)


## 2. Install or update ComfyUI

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import shutil
import subprocess

USE_GOOGLE_DRIVE = True
CLEAN_STALE_FILES = True
WORKSPACE = Path('/content/drive/MyDrive/ComfyUI') if USE_GOOGLE_DRIVE else Path('/content/ComfyUI')
UPSTREAM = 'https://github.com/comfyanonymous/ComfyUI.git'


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


def merge_old_models(backup, workspace):
    old_models = backup / 'models'
    new_models = workspace / 'models'
    if not old_models.exists():
        return
    new_models.mkdir(parents=True, exist_ok=True)
    for old_child in old_models.iterdir():
        target_dir = new_models / old_child.name
        if old_child.is_dir():
            target_dir.mkdir(parents=True, exist_ok=True)
            for item in old_child.iterdir():
                target = target_dir / item.name
                if target.exists():
                    continue
                shutil.move(str(item), str(target))
                print('Restored model file:', target)
        else:
            target = new_models / old_child.name
            if not target.exists():
                shutil.move(str(old_child), str(target))
                print('Restored model file:', target)


def fresh_clone_with_backup(workspace):
    backup = None
    if workspace.exists():
        backup = workspace.with_name('ComfyUI_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
        print('Existing ComfyUI checkout is not updateable. Moving it to:', backup)
        shutil.move(str(workspace), str(backup))
    run(['git', 'clone', '--depth', '1', UPSTREAM, str(workspace)])
    if backup is not None:
        merge_old_models(backup, workspace)


if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')

WORKSPACE.parent.mkdir(parents=True, exist_ok=True)

if not WORKSPACE.exists():
    run(['git', 'clone', '--depth', '1', UPSTREAM, str(WORKSPACE)])
elif not (WORKSPACE / '.git').exists():
    fresh_clone_with_backup(WORKSPACE)

if CLEAN_STALE_FILES and WORKSPACE.exists():
    stale_paths = [
        WORKSPACE / 'comfy_aimdo',
        WORKSPACE / 'comfy_aimdo.py',
        WORKSPACE / 'comfy_api_nodes',
    ]
    for path in stale_paths:
        if path.is_dir():
            shutil.rmtree(path)
            print('Removed stale directory:', path)
        elif path.exists():
            path.unlink()
            print('Removed stale file:', path)

# Try to update the existing checkout. If Git/Drive corruption blocks fetch, rebuild cleanly.
run(['git', 'remote', 'set-url', 'origin', UPSTREAM], cwd=WORKSPACE)
fetch_result = run(['git', 'fetch', '--depth', '1', 'origin'], cwd=WORKSPACE, check=False)
if fetch_result.returncode != 0:
    fresh_clone_with_backup(WORKSPACE)
else:
    branch_result = subprocess.run(['git', 'rev-parse', '--verify', 'origin/master'], cwd=str(WORKSPACE), check=False)
    upstream_branch = 'origin/master' if branch_result.returncode == 0 else 'origin/main'
    run(['git', 'reset', '--hard', upstream_branch], cwd=WORKSPACE)

run(['python3', '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run(['python3', '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121'])
run(['python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=WORKSPACE)
run([
    'python3', '-m', 'pip', 'install', '-q',
    'alembic', 'blake3', 'GitPython', 'accelerate', 'einops', 'transformers',
    'safetensors', 'aiohttp', 'pyyaml', 'Pillow', 'scipy', 'tqdm', 'psutil',
    'tokenizers', 'torchsde', 'kornia', 'spandrel', 'soundfile', 'sentencepiece'
])

print('ComfyUI setup complete:', WORKSPACE)


## 3. Install or update ComfyUI-Manager

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
custom_nodes = WORKSPACE / 'custom_nodes'
manager = custom_nodes / 'ComfyUI-Manager'
custom_nodes.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

if not manager.exists():
    run(['git', 'clone', 'https://github.com/ltdrdata/ComfyUI-Manager.git', str(manager)])

run(['git', 'reset', '--hard', 'HEAD'], cwd=manager)
run(['git', 'pull', '--ff-only'], cwd=manager)
run(['python3', 'custom_nodes/ComfyUI-Manager/cm-cli.py', 'restore-dependencies'], cwd=WORKSPACE)

print('ComfyUI-Manager setup complete')


## 4. Download starter models

In [ ]:
from pathlib import Path
import subprocess

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')
(WORKSPACE / 'models' / 'checkpoints').mkdir(parents=True, exist_ok=True)
(WORKSPACE / 'models' / 'vae').mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

run([
    'wget', '-c',
    'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt',
    '-P', str(WORKSPACE / 'models' / 'checkpoints')
])
run([
    'wget', '-c',
    'https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors',
    '-P', str(WORKSPACE / 'models' / 'vae')
])


## 5. Launch inside Colab + Cloudflare Tunnel

Stay on this Colab page on mobile. The ComfyUI interface will appear below this cell, and a public `trycloudflare.com` link will also be printed.


In [ ]:
from pathlib import Path
from google.colab import output
import re
import socket
import subprocess
import threading
import time

WORKSPACE = Path('/content/drive/MyDrive/ComfyUI')


def run(cmd, cwd=None, check=True):
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


run(['wget', '-q', '-O', '/content/cloudflared-linux-amd64.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared-linux-amd64.deb'], check=False)


def wait_for_port(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            return


def launch_colab_iframe(port):
    wait_for_port(port)
    print('ComfyUI loaded. Showing it inside this Colab page...')
    output.serve_kernel_port_as_iframe(port, height=900)
    print('If the embedded view is small, open this Colab window link:')
    output.serve_kernel_port_as_window(port)


def launch_cloudflared(port):
    wait_for_port(port)
    print('Starting Cloudflare Tunnel...')
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    for line in proc.stderr:
        match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if match:
            print('ComfyUI public URL:', match.group(0))


threading.Thread(target=launch_colab_iframe, daemon=True, args=(8188,)).start()
threading.Thread(target=launch_cloudflared, daemon=True, args=(8188,)).start()
run(['python3', 'main.py', '--dont-print-server'], cwd=WORKSPACE)
